In [ ]:
# CELL 1 - INSTALL + FOLDERS + FIND DATASET
!pip -q install stable-baselines3 gymnasium psutil
import os, glob
ROOT = '/kaggle/working/nfm_experiments'
os.makedirs(ROOT + '/nfm', exist_ok=True)
os.makedirs(ROOT + '/results', exist_ok=True)
cands = glob.glob('/kaggle/input/**/Captured_packets.csv', recursive=True)
assert cands, 'Captured_packets.csv not found: add it as a Dataset input (Add Input)'
CSV = cands[0]
print('Dataset:', CSV)

In [ ]:
%%writefile /kaggle/working/nfm_experiments/nfm/__init__.py
# package


In [ ]:
%%writefile /kaggle/working/nfm_experiments/nfm/common.py
"""Shared utilities: seeding, software/hardware report, confidence intervals."""
import json
import os
import platform
import random
import sys

import numpy as np


def set_seed(seed: int) -> None:
    """Seed python, numpy and (if installed) torch. Each experiment documents which
    seed controls what: data generation, model training and evaluation use separate
    seed ranges (see README, 'Seeds')."""
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def env_report(path=None) -> dict:
    """Software versions and hardware, for the reproducibility section."""
    info = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "processor": platform.processor(),
        "cpu_count": os.cpu_count(),
    }
    for m in ["numpy", "pandas", "sklearn", "xgboost", "torch", "gymnasium",
              "stable_baselines3", "scipy", "matplotlib"]:
        try:
            mod = __import__(m)
            info[m] = getattr(mod, "__version__", "?")
        except Exception:
            info[m] = None
    try:
        import torch
        info["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    except Exception:
        info["gpu"] = None
    try:
        import psutil
        info["ram_gb"] = round(psutil.virtual_memory().total / 2 ** 30, 1)
    except Exception:
        info["ram_gb"] = None
    if path:
        with open(path, "w") as f:
            json.dump(info, f, indent=2)
    return info


def wilson(k: int, n: int, z: float = 1.96):
    """Wilson 95% interval for a proportion k/n."""
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return float((c - h) / d), float((c + h) / d)


def bootstrap_ci(metric_fn, *arrays, n_boot=1000, seed=0, groups=None, alpha=0.05):
    """Percentile bootstrap CI. If `groups` is given, whole groups (e.g. flows) are
    resampled (cluster bootstrap), which respects dependence between packets of a flow."""
    arrays = [np.asarray(a) for a in arrays]
    rng = np.random.default_rng(seed)
    n = len(arrays[0])
    vals = []
    if groups is None:
        for _ in range(n_boot):
            idx = rng.integers(0, n, n)
            try:
                vals.append(metric_fn(*[a[idx] for a in arrays]))
            except ValueError:
                pass
    else:
        groups = np.asarray(groups)
        ug, inv = np.unique(groups, return_inverse=True)
        members = [np.where(inv == g)[0] for g in range(len(ug))]
        for _ in range(n_boot):
            pick = rng.integers(0, len(ug), len(ug))
            idx = np.concatenate([members[g] for g in pick])
            try:
                vals.append(metric_fn(*[a[idx] for a in arrays]))
            except ValueError:
                pass
    if not vals:
        return float("nan"), float("nan")
    lo, hi = np.percentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)


def paired_bootstrap_diff(a, b, n_boot=5000, seed=0):
    """Mean(a-b) with 95% CI over paired per-episode values (same initial states)."""
    d = np.asarray(a, float) - np.asarray(b, float)
    rng = np.random.default_rng(seed)
    means = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)]
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(d.mean()), float(lo), float(hi)


In [ ]:
%%writefile /kaggle/working/nfm_experiments/nfm/sim.py
"""Graph-based network simulator.

One simulator defines the *common network state* that every agent reads, which
addresses the reviewer's point that detection / diagnosis / recovery must operate
on a consistently defined state:

    state X : (n_nodes, 7) array, columns = METRICS, values in [0, 1]
    topology: undirected graph (adjacency matrix)
    faults  : root node(s) + fault type + magnitude, propagated to neighbours

Everything here is synthetic and is documented as such in the paper.
"""
import numpy as np

METRICS = ["cpu", "memory", "bandwidth", "latency", "packet_loss", "jitter", "throughput"]
CPU, MEM, BW, LAT, LOSS, JIT, THR = range(7)
ACTIONS = ["no_action", "reroute", "restart", "failover", "reconfigure"]
ACTION_COST = np.array([0.0, 0.01, 0.05, 0.03, 0.02])
FAULT_TYPES = ["cpu_overload", "memory_leak", "link_congestion",
               "packet_loss_burst", "latency_spike"]

# Health = 1 - weighted sum of "bad" metrics (same weights as the original paper code;
# throughput is observed but carries no health weight).
HEALTH_W = np.array([0.15, 0.10, 0.15, 0.25, 0.20, 0.10, 0.0])
BASE_MEAN = np.array([0.35, 0.40, 0.40, 0.20, 0.03, 0.10, 0.60])
BASE_STD = 0.05
NODE_LOAD_STD = 0.08            # per-node heterogeneity of baseline cpu / bandwidth
CRIT = np.array([0.90, 0.90, 0.95, 0.85, 0.80, 0.80, 2.0])   # critical levels (thr never)
HEALTHY_DEFICIT = 0.04          # node counts as recovered if deficit <= this
ABNORMAL_DEFICIT = 0.06         # node counts as abnormal (ground truth) if deficit > this
RULE_HEALTH_THR = 0.72          # what a naive observer (no per-node reference) would use

# Fault signatures: delta added to the root node = signature * magnitude
SIGNATURE = {
    "cpu_overload":      np.array([1.0, 0.2, 0.0, 0.3, 0.0, 0.1, -0.3]),
    "memory_leak":       np.array([0.3, 1.0, 0.0, 0.2, 0.0, 0.0, -0.1]),
    "link_congestion":   np.array([0.0, 0.0, 1.0, 0.6, 0.3, 0.4, -0.4]),
    "packet_loss_burst": np.array([0.0, 0.0, 0.1, 0.2, 1.0, 0.3, -0.4]),
    "latency_spike":     np.array([0.1, 0.0, 0.0, 1.0, 0.0, 0.5, -0.2]),
}


def make_topology(n, kind, rng, extra=0.15):
    """Connected random topology. kind: 'random' (random tree), 'ba' (preferential
    attachment tree), 'ring' (ring + chords). `extra*n` additional random edges."""
    edges = set()
    deg = np.zeros(n)
    if kind == "ring" and n >= 3:
        for i in range(n):
            j = (i + 1) % n
            edges.add((min(i, j), max(i, j)))
    else:
        for i in range(1, n):
            if kind == "ba":
                w = deg[:i] + 1.0
                j = int(rng.choice(i, p=w / w.sum()))
            else:
                j = int(rng.integers(0, i))
            edges.add((j, i))
            deg[i] += 1
            deg[j] += 1
    for _ in range(int(extra * n)):
        a, b = rng.integers(0, n, 2)
        if a != b:
            edges.add((int(min(a, b)), int(max(a, b))))
    return np.array(sorted(edges), dtype=int).reshape(-1, 2)


class Network:
    def __init__(self, n, rng, topo="random", extra=0.15, noise=0.0, p_missing=0.0):
        self.n, self.rng = n, rng
        self.noise, self.p_missing = noise, p_missing
        self.edges = make_topology(n, topo, rng, extra)
        self.adj = np.zeros((n, n), dtype=bool)
        for a, b in self.edges:
            self.adj[a, b] = self.adj[b, a] = True
        base = BASE_MEAN + rng.normal(0, BASE_STD, (n, 7))
        base[:, [CPU, BW]] += rng.normal(0, NODE_LOAD_STD, (n, 2))
        self.X = np.clip(base, 0, 1)
        self.base = self.X.copy()                        # healthy operating point
        self.ref_health = self.health(self.X)           # per-node healthy reference
        self.roots, self.ftypes = [], []
        self.prop, self.prop_d, self.induced = 0.4, {}, {}   # fault propagation bookkeeping
        self.pending = np.zeros((n, 7))                  # transient (restart downtime)
        self.t = 0

    # ---------------------------------------------------------------- basics
    def clone(self):
        c = Network.__new__(Network)
        c.__dict__.update(self.__dict__)
        c.X, c.pending = self.X.copy(), self.pending.copy()
        c.roots, c.ftypes = list(self.roots), list(self.ftypes)
        c.induced = {r: m.copy() for r, m in self.induced.items()}
        return c

    def neighbors(self, v):
        return np.flatnonzero(self.adj[v])

    def degree(self, v):
        return int(self.adj[v].sum())

    def health(self, X=None):
        X = self.X if X is None else X
        return 1.0 - X @ HEALTH_W

    def deficit(self, X=None):
        """True per-node deficit w.r.t. the node's own healthy reference (ground truth)."""
        return self.ref_health - self.health(X)

    def abnormal_nodes(self):
        return self.deficit() > ABNORMAL_DEFICIT

    def recovered(self):
        return bool((self.deficit() <= HEALTHY_DEFICIT).all())

    def hop_dist(self, v):
        d = np.full(self.n, -1)
        d[v] = 0
        frontier = [v]
        while frontier:
            nxt = []
            for u in frontier:
                for w in np.flatnonzero(self.adj[u]):
                    if d[w] < 0:
                        d[w] = d[u] + 1
                        nxt.append(w)
            frontier = nxt
        return d

    # ---------------------------------------------------------------- faults
    def inject(self, node, ftype, mag, prop=0.4):
        """Add a fault at `node`. Its effect propagates to other nodes with factor
        prop**hops and *follows the root*: when the root heals, induced effects fade."""
        self.prop = prop
        self.X[node] += SIGNATURE[ftype] * mag
        np.clip(self.X, 0, 1, out=self.X)
        self.roots.append(int(node))
        self.ftypes.append(ftype)
        self.prop_d[int(node)] = self.hop_dist(node)
        self.induced[int(node)] = np.zeros((self.n, 7))
        self._refresh_induced()

    def _refresh_induced(self):
        for r in self.roots:
            delta = self.X[r] - self.base[r]
            delta = np.where(np.arange(7) == THR, np.minimum(delta, 0), np.maximum(delta, 0))
            d = self.prop_d[r]
            new = np.zeros((self.n, 7))
            for w in range(self.n):
                if d[w] > 0:
                    new[w] = delta * (self.prop ** d[w])
            self.X += new - self.induced[r]
            self.induced[r] = new
        np.clip(self.X, 0, 1, out=self.X)

    def inject_random(self, k=1, mag_range=(0.35, 0.60), ftypes=None):
        ftypes = ftypes or FAULT_TYPES
        nodes = self.rng.choice(self.n, size=min(k, self.n), replace=False)
        for v in nodes:
            self.inject(int(v), str(self.rng.choice(ftypes)), float(self.rng.uniform(*mag_range)))

    # ---------------------------------------------------------------- telemetry
    def telemetry(self):
        """What the agents see: noisy, possibly missing (imputed by column mean)."""
        obs = self.X + self.rng.normal(0, self.noise, self.X.shape) if self.noise > 0 else self.X.copy()
        if self.p_missing > 0:
            miss = self.rng.random(self.X.shape) < self.p_missing
            masked = np.where(miss, np.nan, obs)
            ok = (~miss).sum(0) > 0
            colmean = BASE_MEAN.copy()
            colmean[ok] = np.nansum(masked[:, ok], axis=0) / (~miss[:, ok]).sum(0)
            obs = np.where(miss, colmean, obs)
        return np.clip(obs, 0, 1)

    # ---------------------------------------------------------------- actions
    def available(self, v):
        """Which recovery actions are *available* at node v given the topology/state."""
        nb = self.neighbors(v)
        av = np.ones(5, dtype=bool)
        av[1] = len(nb) >= 2                                   # reroute needs an alternative path
        av[3] = bool(len(nb)) and bool(((self.X[nb, CPU] < 0.85) & (self.X[nb, BW] < 0.85)).any())
        return av

    def context(self, v):
        """Topology context handed to the shield / recovery agent."""
        nb = self.neighbors(v)
        if len(nb):
            w = nb[np.argmin(self.X[nb, CPU] + self.X[nb, BW])]
            post_cpu, post_bw = self.X[w, CPU] + 0.08, self.X[w, BW] + 0.08
        else:
            post_cpu = post_bw = 1.0
        return {"degree": len(nb), "available": self.available(v),
                "failover_target_post_cpu": float(post_cpu),
                "failover_target_post_bw": float(post_bw)}

    def apply(self, v, a, deterministic=False):
        """Apply action a at node v. deterministic=True disables noise and drift (planning)."""
        X = self.X
        X -= self.pending                    # transient effects of the previous step expire
        self.pending = np.zeros_like(self.pending)
        nb = self.neighbors(v)
        valid_id = isinstance(a, (int, np.integer)) and 0 <= a < 5
        invalid = (not valid_id) or (not self.available(v)[a])
        if invalid:
            if a != 0 or not valid_id:                        # disruptive misconfiguration
                X[v, LAT] += 0.15
                X[v, THR] *= 0.8
        elif a == 1:      # reroute
            X[v, LAT] *= 0.75; X[v, LOSS] *= 0.60; X[v, BW] *= 0.85; X[v, JIT] *= 0.85
            X[nb, BW] += 0.04; X[nb, CPU] += 0.02
        elif a == 2:      # restart (downtime is transient; neighbours feel the cascade)
            X[v, CPU] *= 0.40; X[v, MEM] *= 0.40; X[v, LAT] *= 0.85
            down = np.zeros_like(self.pending)
            down[v, LAT] += 0.12
            down[v, THR] -= 0.30
            casc = 0.12 * min(len(nb), 6) / 6.0
            down[nb, LAT] += casc
            X += down
            self.pending = down
        elif a == 3:      # failover (shifts load to the least loaded neighbour)
            w = nb[np.argmin(X[nb, CPU] + X[nb, BW])]
            X[v, LAT] *= 0.60; X[v, LOSS] *= 0.50; X[v, CPU] *= 0.85; X[v, BW] *= 0.85
            X[w, CPU] += 0.08; X[w, BW] += 0.08
        elif a == 4:      # reconfigure
            X[v, BW] *= 0.60; X[v, JIT] *= 0.60; X[v, LAT] *= 0.90
        if not deterministic:
            X += self.rng.normal(0, 0.01, X.shape)
            for r, ft in zip(self.roots, self.ftypes):       # memory leak keeps growing
                if ft == "memory_leak" and X[r, MEM] > 0.55:
                    X[r, MEM] += 0.02
        np.clip(X, 0, 1, out=X)
        self._refresh_induced()
        self.t += 1

    def consequence(self, v, a):
        """Ground-truth consequence of action a at node v, measured in the environment
        (independent of any shield rule). 'harm' = invalid/disruptive action, a metric
        that newly crosses its critical level anywhere, collateral health loss on another
        node > 0.03, or a >0.02 health loss on the target itself."""
        valid_id = isinstance(a, (int, np.integer)) and 0 <= a < 5
        invalid = (not valid_id) or (not self.available(v)[a])
        X0, h0 = self.X.copy(), self.health()
        c = self.clone()
        c.apply(v, a, deterministic=True)
        h1 = c.health()
        others = np.arange(self.n) != v
        collateral = float(max(0.0, (h0 - h1)[others].max())) if self.n > 1 else 0.0
        crit_new = bool(((c.X >= CRIT) & (X0 < CRIT)).any())
        dt = float(h1[v] - h0[v])
        harm = bool((invalid and a != 0) or crit_new or collateral > 0.03 or dt < -0.02)
        return {"harm": harm, "invalid": bool(invalid and a != 0), "crit_new": crit_new,
                "collateral": collateral, "dh_target": dt, "dh_net": float(h1.mean() - h0.mean())}


In [ ]:
%%writefile /kaggle/working/nfm_experiments/nfm/shield.py
"""Safety & Verification layer and its evaluation.

ShieldV1  = the rule used in the original paper (fixed increments, threshold 0.70).
ShieldV2  = topology-aware extension (action availability, neighbour headroom, fan-out).

Evaluation is against *environment consequences* (Network.consequence), not against the
shield's own rule: the shield is treated as a binary classifier of harmful actions.
Run:  python -m nfm.shield --n_cases 20000 --out results/shield
"""
import argparse
import os

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from .common import set_seed
from .sim import (ACTIONS, BW, CPU, LAT, LOSS, MEM, Network)

GOOD = {"cpu_overload": [2], "memory_leak": [2], "link_congestion": [4, 3, 1],
        "packet_loss_burst": [1, 3], "latency_spike": [3, 1]}


def valid_action(a):
    return isinstance(a, (int, np.integer)) and 0 <= a < 5


class ShieldV1:
    """Rule from the paper: +0.20 per threshold violation, +0.10 restart, +0.05 failover."""
    name = "v1_paper_rule"

    def __init__(self, max_risk=0.70):
        self.max_risk = max_risk

    def risk(self, x, a, ctx=None):
        if not valid_action(a):
            return 1.0
        r = 0.0
        r += 0.20 * (x[CPU] > 0.90) + 0.20 * (x[MEM] > 0.90) + 0.20 * (x[BW] > 0.90)
        r += 0.20 * (x[LAT] > 0.80) + 0.20 * (x[LOSS] > 0.80)
        r += 0.10 * (a == 2) + 0.05 * (a == 3)
        return float(min(r, 1.0))

    def validate(self, x, a, ctx=None):
        r = self.risk(x, a, ctx)
        return r <= self.max_risk, r


class ShieldV2(ShieldV1):
    """V1 + topology/action-specific terms. Increments are documented here and their
    empirical basis is reported by calibrate_increments()."""
    name = "v2_topology_aware"

    def __init__(self, max_risk=0.50):
        super().__init__(max_risk)

    def risk(self, x, a, ctx=None):
        r = super().risk(x, a, ctx)
        if not valid_action(a) or ctx is None:
            return r
        if a != 0 and not ctx["available"][a]:
            r += 0.80                                   # action not available at this node
        if a == 3 and (ctx["failover_target_post_cpu"] > 0.90 or ctx["failover_target_post_bw"] > 0.95):
            r += 0.45                                   # failover would overload the neighbour
        if a == 2 and ctx["degree"] >= 4:
            r += 0.15                                   # restart of a high fan-out node (cascade)
        return float(min(r, 1.0))


# --------------------------------------------------------------------------- test cases
def sample_cases(n_cases, seed, loaded_p=0.4):
    rng = np.random.default_rng(seed)
    kinds = ["random_valid", "random_any", "disruptive", "state_inappropriate", "invalid"]
    rows = []
    for _ in range(n_cases):
        n = int(rng.integers(6, 15))
        net = Network(n, rng, topo=str(rng.choice(["random", "ba", "ring"])), noise=0.02)
        net.inject_random(int(rng.integers(1, 3)))
        v = net.roots[0] if rng.random() < 0.7 else int(rng.integers(0, n))
        if rng.random() < loaded_p:                       # congested neighbourhood
            nb = net.neighbors(v)
            net.X[nb, CPU] = np.clip(net.X[nb, CPU] + rng.uniform(0.2, 0.5, len(nb)), 0, 1)
            net.X[nb, BW] = np.clip(net.X[nb, BW] + rng.uniform(0.2, 0.5, len(nb)), 0, 1)
        kind = str(rng.choice(kinds))
        av = net.available(v)
        if kind == "random_valid":
            a = int(rng.choice(np.flatnonzero(av)))
        elif kind == "random_any":
            a = int(rng.integers(0, 5))
        elif kind == "disruptive":
            a = int(rng.choice([2, 3]))
        elif kind == "state_inappropriate":
            good = set(GOOD[net.ftypes[0]]) if v == net.roots[0] else set()
            a = int(rng.choice([x for x in range(1, 5) if x not in good]))
        else:  # invalid: unavailable action if one exists, otherwise an out-of-range id
            bad = np.flatnonzero(~av)
            a = int(rng.choice(bad)) if len(bad) else 7
        cons = net.consequence(v, a)
        rows.append({"kind": kind, "action": a, "x": net.telemetry()[v], "ctx": net.context(v),
                     "useless": bool(cons["dh_target"] < 0.01), **cons})
    return rows


def risks(shield, cases):
    return np.array([shield.risk(c["x"], c["action"], c["ctx"]) for c in cases])


def evaluate_shield(shield, cases, thr=None):
    thr = shield.max_risk if thr is None else thr
    r = risks(shield, cases)
    harm = np.array([c["harm"] for c in cases])
    rej = r > thr
    tp, fp = int((rej & harm).sum()), int((rej & ~harm).sum())
    fn, tn = int((~rej & harm).sum()), int((~rej & ~harm).sum())
    out = {"shield": shield.name, "threshold": thr, "n": len(cases), "harm_rate": float(harm.mean()),
           "recall_harm": tp / max(tp + fn, 1), "fpr": fp / max(fp + tn, 1),
           "precision": tp / max(tp + fp, 1), "TP": tp, "FP": fp, "FN": fn, "TN": tn,
           "auc": roc_auc_score(harm, r) if 0 < harm.sum() < len(harm) else float("nan")}
    return out


def per_kind(shield, cases):
    r = risks(shield, cases)
    df = pd.DataFrame({"kind": [c["kind"] for c in cases], "harm": [c["harm"] for c in cases],
                       "useless": [c["useless"] for c in cases], "rejected": r > shield.max_risk})
    g = df.groupby("kind").agg(n=("harm", "size"), harm_rate=("harm", "mean"),
                               useless_rate=("useless", "mean"), reject_rate=("rejected", "mean"))
    g["harm_caught"] = df[df.harm].groupby("kind")["rejected"].mean()
    return g.reset_index()


def threshold_sweep(shield, cases, grid=np.arange(0.1, 1.0, 0.1)):
    return pd.DataFrame([evaluate_shield(shield, cases, t) for t in grid])


def calibrate_threshold(shield, cases, max_fpr=0.05):
    """Empirical basis for the threshold: highest recall subject to FPR <= max_fpr."""
    best, best_rec = shield.max_risk, -1
    for t in np.arange(0.05, 1.0, 0.05):
        m = evaluate_shield(shield, cases, t)
        if m["fpr"] <= max_fpr and m["recall_harm"] > best_rec:
            best, best_rec = float(t), m["recall_harm"]
    return best


def calibrate_increments(cases):
    """Logistic regression of environment harm on the shield's indicator features.
    Coefficients show which risk terms carry signal; they are the empirical basis (or the
    lack of one) for the hand-set increments."""
    feats, y = [], []
    for c in cases:
        x, a, ctx = c["x"], c["action"], c["ctx"]
        ok = valid_action(a)
        feats.append([x[CPU] > .9, x[MEM] > .9, x[BW] > .9, x[LAT] > .8, x[LOSS] > .8,
                      ok and a == 2, ok and a == 3,
                      ok and a != 0 and not ctx["available"][a],
                      ok and a == 3 and (ctx["failover_target_post_cpu"] > .9 or ctx["failover_target_post_bw"] > .95),
                      ok and a == 2 and ctx["degree"] >= 4, not ok])
        y.append(c["harm"])
    names = ["cpu>.9", "mem>.9", "bw>.9", "lat>.8", "loss>.8", "restart", "failover",
             "unavailable", "failover_overload", "restart_fanout", "invalid_id"]
    clf = LogisticRegression(max_iter=1000, C=1.0).fit(np.array(feats, float), np.array(y))
    return pd.DataFrame({"feature": names, "logit_coef": clf.coef_[0],
                         "prevalence": np.array(feats, float).mean(0)})


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n_cases", type=int, default=20000)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="results/shield")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    set_seed(a.seed)
    calib = sample_cases(a.n_cases // 2, 10_000 + a.seed)       # calibration cases
    test = sample_cases(a.n_cases, 20_000 + a.seed)             # disjoint test cases
    v1, v2 = ShieldV1(), ShieldV2()
    v2_cal = ShieldV2(max_risk=calibrate_threshold(v2, calib))
    rows = [evaluate_shield(s, test) for s in (v1, v2, v2_cal)]
    summary = pd.DataFrame(rows)
    summary.to_csv(f"{a.out}/summary.csv", index=False)
    per_kind(v1, test).assign(shield=v1.name).to_csv(f"{a.out}/per_kind_v1.csv", index=False)
    per_kind(v2_cal, test).assign(shield=v2.name).to_csv(f"{a.out}/per_kind_v2.csv", index=False)
    threshold_sweep(v1, test).to_csv(f"{a.out}/sweep_v1.csv", index=False)
    threshold_sweep(v2, test).to_csv(f"{a.out}/sweep_v2.csv", index=False)
    calibrate_increments(calib).to_csv(f"{a.out}/increment_logit.csv", index=False)
    print(summary.round(3).to_string(index=False))
    print("\nPer injected-action type (V1, paper rule, thr 0.70):")
    print(per_kind(v1, test).round(3).to_string(index=False))
    print("\nPer injected-action type (V2, calibrated threshold %.2f):" % v2_cal.max_risk)
    print(per_kind(v2_cal, test).round(3).to_string(index=False))
    print("\nLogistic coefficients of environment harm on shield features:")
    print(calibrate_increments(calib).round(3).to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/nfm_experiments/nfm/recovery.py
"""Recovery Agent experiments.

Addresses: PPO collapse, reward design, training budget, hyper-parameter tuning, multiple
seeds, stronger baselines (rule-based AND optimisation-based), action distributions and
per-fault-type results.

Run:  python -m nfm.recovery --out results/recovery            (full)
      python -m nfm.recovery --out results/recovery --fast      (smoke test)
      python -m nfm.recovery --out results/recovery --skip_ppo  (baselines only)
"""
import argparse
import itertools
import os

import numpy as np
import pandas as pd

from .common import paired_bootstrap_diff, set_seed, wilson
from .sim import (ACTION_COST, ACTIONS, BW, CPU, FAULT_TYPES, HEALTHY_DEFICIT, LAT, LOSS, MEM,
                  Network)

OBS_DIM = 17


class RecoveryCore:
    """Single-fault recovery episode on a random topology. Pure numpy (no gym needed).

    Observation (17) : telemetry of the target node (7), mean telemetry of its neighbours (7),
                       degree/10, failover-available flag, remaining-steps fraction.
    Actions          : 0 no-op, 1 reroute, 2 restart, 3 failover, 4 reconfigure
                       (unavailable actions are executed as disruptive misconfigurations).
    reward_mode      : 'legacy'  -> reward = node health every step (as in the original script)
                       'shaped'  -> 10*(drop in network deficit) - 2*action_cost - 0.02/step
                                    + 1.0 on recovery - 0.5 for unavailable action
    loc_err          : probability that the diagnosed target is a wrong node (diagnosis error).
    """

    def __init__(self, n_range=(6, 12), max_steps=10, reward_mode="shaped", loc_err=0.0,
                 noise=0.02, p_missing=0.0, mag_range=(0.35, 0.60), ftypes=None, terminate=None):
        self.n_range, self.max_steps, self.reward_mode = n_range, max_steps, reward_mode
        self.loc_err, self.noise, self.p_missing = loc_err, noise, p_missing
        self.mag_range, self.ftypes = mag_range, ftypes
        self.terminate = (reward_mode == "shaped") if terminate is None else terminate

    def reset(self, seed):
        rng = np.random.default_rng(seed)
        n = int(rng.integers(*self.n_range))
        self.net = Network(n, rng, topo=str(rng.choice(["random", "ba", "ring"])),
                           noise=self.noise, p_missing=self.p_missing)
        self.net.inject_random(1, self.mag_range, self.ftypes)
        self.root, self.ftype = self.net.roots[0], self.net.ftypes[0]
        self.target = self.root
        if rng.random() < self.loc_err and n > 1:
            self.target = int(rng.choice([v for v in range(n) if v != self.root]))
        self.i = 0
        self.d0 = float(self.net.deficit()[self.root])
        return self._obs()

    def _obs(self):
        tel, v = self.net.telemetry(), self.target
        nb = self.net.neighbors(v)
        nbm = tel[nb].mean(0) if len(nb) else tel[v]
        ctx = self.net.context(v)
        return np.concatenate([tel[v], nbm, [min(ctx["degree"], 10) / 10.0,
                                             float(ctx["available"][3]),
                                             (self.max_steps - self.i) / self.max_steps]]).astype(np.float32)

    def step(self, a):
        a, net, v = int(a), self.net, self.target
        cons = net.consequence(v, a)
        h0, nd0 = net.health()[v], net_deficit(net)
        net.apply(v, a)
        h1, nd1 = net.health()[v], net_deficit(net)
        recovered = net.recovered()
        self.i += 1
        if self.reward_mode == "legacy":
            reward = float(h1)
        else:
            reward = 10 * (nd0 - nd1) - 2 * ACTION_COST[a] - 0.02 + (1.0 if recovered else 0.0) \
                - (0.5 if cons["invalid"] else 0.0)
        terminated = bool(recovered and self.terminate)
        truncated = (not terminated) and self.i >= self.max_steps
        return self._obs(), float(reward), terminated, truncated, {"recovered": recovered,
                                                                    "harm": cons["harm"]}


# --------------------------------------------------------------------------- policies
class RandomPolicy:
    name = "random"

    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)

    def rank(self, obs, core):
        return list(self.rng.permutation(5))

    def act(self, obs, core):
        return int(self.rank(obs, core)[0])


class NoActionPolicy:
    name = "no_action"

    def rank(self, obs, core):
        return [0, 1, 2, 3, 4]

    def act(self, obs, core):
        return 0


class HeuristicPolicy:
    """Rule-based policy, fully specified:
       cpu>0.60 or mem>0.60 -> restart | else bw>0.60 -> reconfigure |
       else loss>0.10 or lat>0.33 or jitter>0.30 -> failover (if a neighbour has headroom)
       else reroute | else no-op"""
    name = "heuristic_rules"

    def rank(self, obs, core):
        x, headroom = obs, obs[15] > 0.5
        if x[CPU] > 0.60 or x[MEM] > 0.60:
            first = 2
        elif x[BW] > 0.60:
            first = 4
        elif x[LOSS] > 0.10 or x[LAT] > 0.33 or x[5] > 0.30:
            first = 3 if headroom else 1
        else:
            first = 0
        return [first] + [a for a in (1, 2, 3, 4, 0) if a != first]

    def act(self, obs, core):
        return int(self.rank(obs, core)[0])


def net_deficit(net):
    """Network-level badness: sum of positive per-node deficits (counts collateral damage)."""
    return float(np.maximum(net.deficit(), 0).sum())


def _score(net, v, a):
    """Model-based one-step score used by the optimisation baselines (uses the simulator)."""
    if not net.available(v)[a] and a != 0:
        return -1e9
    c = net.clone()
    c.apply(v, a, deterministic=True)
    return 10 * (net_deficit(net) - net_deficit(c)) - 2 * ACTION_COST[a], c


class GreedyOraclePolicy:
    """Optimisation baseline: argmax of one-step model-based score (needs the simulator)."""
    name = "greedy_model_based"

    def rank(self, obs, core):
        sc = [(_score(core.net, core.target, a)[0] if (core.net.available(core.target)[a] or a == 0)
               else -1e9) for a in range(5)]
        return list(np.argsort(-np.array(sc)))

    def act(self, obs, core):
        return int(self.rank(obs, core)[0])


class Lookahead2Policy:
    """Optimisation baseline: exhaustive two-step search on the simulator model."""
    name = "lookahead2_model_based"

    def rank(self, obs, core):
        net, v = core.net, core.target
        best = []
        for a1 in range(5):
            if a1 != 0 and not net.available(v)[a1]:
                best.append(-1e9)
                continue
            s1, c1 = _score(net, v, a1)
            if c1.recovered():
                best.append(s1)
                continue
            s2s = [_score(c1, v, a2)[0] for a2 in range(5) if (a2 == 0 or c1.available(v)[a2])]
            best.append(s1 + 0.9 * max(s2s))     # discount: prefer acting now to waiting
        return list(np.argsort(-np.array(best)))

    def act(self, obs, core):
        return int(self.rank(obs, core)[0])


class PPOPolicy:
    def __init__(self, model, name="ppo"):
        self.model, self.name = model, name

    def rank(self, obs, core=None):
        import torch
        with torch.no_grad():
            t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0).to(self.model.device)
            probs = self.model.policy.get_distribution(t).distribution.probs[0].cpu().numpy()
        return list(np.argsort(-probs))

    def act(self, obs, core=None):
        return int(self.rank(obs, core)[0])


# --------------------------------------------------------------------------- evaluation
def run_episodes(policy, seeds, **core_kwargs):
    """Evaluate a policy on episodes whose initial state is fixed by the episode seed
    (identical across policies). Evaluation always uses the shaped reward."""
    core_kwargs = {**core_kwargs, "reward_mode": "shaped"}
    rows = []
    for s in seeds:
        core = RecoveryCore(**core_kwargs)
        obs = core.reset(s)
        total, acts, harm, rec = 0.0, [], 0, None
        for t in range(core.max_steps):
            a = policy.act(obs, core)
            obs, r, term, trunc, info = core.step(a)
            total += r
            acts.append(a)
            harm += int(info["harm"])
            if info["recovered"]:
                rec = t + 1
                break
        d_final = float(core.net.deficit()[core.root])
        rows.append({"seed": s, "ftype": core.ftype, "success": rec is not None,
                     "success_half": (core.d0 - d_final) >= 0.5 * core.d0, "steps": rec,
                     "reward": total, "harm": harm, "actions": acts, "d0": core.d0})
    return pd.DataFrame(rows)


def summarize(df, name):
    acts = np.concatenate([np.array(a) for a in df["actions"]])
    share = np.bincount(acts, minlength=5) / len(acts)
    ent = float(-(share[share > 0] * np.log(share[share > 0])).sum())
    k, n = int(df["success"].sum()), len(df)
    lo, hi = wilson(k, n)
    return {"policy": name, "success": k / n, "ci_lo": lo, "ci_hi": hi,
            "success_half": df["success_half"].mean(),
            "steps_to_recover": df["steps"].dropna().mean(), "reward": df["reward"].mean(),
            "harmful_actions_per_ep": df["harm"].mean(), "dominant_action": ACTIONS[int(share.argmax())],
            "dominant_share": float(share.max()), "action_entropy": ent,
            **{f"share_{a}": float(s) for a, s in zip(ACTIONS, share)}}


def per_fault(df, name):
    g = df.groupby("ftype").agg(n=("success", "size"), success=("success", "mean"),
                                steps=("steps", "mean"), harm=("harm", "mean")).reset_index()
    g.insert(0, "policy", name)
    return g


# --------------------------------------------------------------------------- PPO
def make_gym_env(**kw):
    import gymnasium as gym
    from gymnasium import spaces

    class RecoveryEnv(gym.Env):
        def __init__(self):
            super().__init__()
            self.core = RecoveryCore(**kw)
            self.observation_space = spaces.Box(-1.0, 2.0, (OBS_DIM,), np.float32)
            self.action_space = spaces.Discrete(5)

        def reset(self, seed=None, options=None):
            super().reset(seed=seed)
            return self.core.reset(int(self.np_random.integers(0, 2 ** 31 - 1))), {}

        def step(self, action):
            return self.core.step(int(action))

    return RecoveryEnv()


def train_ppo(seed, timesteps, hp, reward_mode="shaped", **env_kw):
    from stable_baselines3 import PPO
    set_seed(seed)
    env = make_gym_env(reward_mode=reward_mode, **env_kw)
    model = PPO("MlpPolicy", env, seed=seed, device="cpu", verbose=0, batch_size=64,
                policy_kwargs={"net_arch": [64, 64]}, **hp)
    model.learn(total_timesteps=timesteps)
    return model


def tune(grid, timesteps, val_seeds, **env_kw):
    rows = []
    for lr, ent, gamma, nsteps in itertools.product(*grid):
        hp = dict(learning_rate=lr, ent_coef=ent, gamma=gamma, n_steps=nsteps)
        m = train_ppo(12345, timesteps, hp, **env_kw)
        df = run_episodes(PPOPolicy(m), val_seeds, **env_kw)
        rows.append({**hp, "val_success": df["success"].mean(), "val_reward": df["reward"].mean()})
        print("tune", rows[-1])
    t = pd.DataFrame(rows).sort_values(["val_success", "val_reward"], ascending=False)
    best = t.iloc[0]
    return dict(learning_rate=float(best.learning_rate), ent_coef=float(best.ent_coef),
                gamma=float(best.gamma), n_steps=int(best.n_steps)), t


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="results/recovery")
    ap.add_argument("--seeds", type=int, default=5)
    ap.add_argument("--timesteps", type=int, default=200_000)
    ap.add_argument("--episodes", type=int, default=200)
    ap.add_argument("--tune_timesteps", type=int, default=50_000)
    ap.add_argument("--no_tune", action="store_true")
    ap.add_argument("--skip_ppo", action="store_true")
    ap.add_argument("--fast", action="store_true")
    ap.add_argument("--loc_err", type=float, default=0.0)
    a = ap.parse_args()
    if a.fast:
        a.seeds, a.timesteps, a.episodes, a.tune_timesteps = 2, 20_000, 50, 10_000
    os.makedirs(a.out, exist_ok=True)
    env_kw = {"loc_err": a.loc_err}
    test_seeds = list(range(100_000, 100_000 + a.episodes))     # disjoint from training/val
    val_seeds = list(range(200_000, 200_050))

    results = {}
    for pol in [NoActionPolicy(), RandomPolicy(0), HeuristicPolicy(), GreedyOraclePolicy(),
                Lookahead2Policy()]:
        results[pol.name] = run_episodes(pol, test_seeds, **env_kw)

    ppo_runs = {}
    if not a.skip_ppo:
        if a.no_tune or a.fast:
            hp = dict(learning_rate=3e-4, ent_coef=0.01, gamma=0.95, n_steps=1024)
        else:
            grid = ([3e-4, 1e-3], [0.0, 0.01, 0.03], [0.95, 0.99], [1024])
            hp, tune_table = tune(grid, a.tune_timesteps, val_seeds, **env_kw)
            tune_table.to_csv(f"{a.out}/ppo_tuning.csv", index=False)
        print("PPO hyper-parameters:", hp)
        for mode in ["legacy", "shaped"]:         # legacy reward = reward-function ablation
            for s in range(a.seeds):
                m = train_ppo(s, a.timesteps, hp, reward_mode=mode, **env_kw)
                if mode == "shaped" and s == 0:      # reused by nfm.integrated via --ppo_path
                    m.save(f"{a.out}/ppo_shaped_seed0")
                ppo_runs[(mode, s)] = run_episodes(PPOPolicy(m), test_seeds, **env_kw)

    rows, pf, dist = [], [], []
    for name, df in results.items():
        rows.append(summarize(df, name)); pf.append(per_fault(df, name))
    ppo_summ = []
    for mode in ["legacy", "shaped"]:
        runs = [df for (m, s), df in ppo_runs.items() if m == mode]
        if not runs:
            continue
        per_seed = pd.DataFrame([summarize(df, f"ppo_{mode}_seed{s}")
                                 for (m, s), df in ppo_runs.items() if m == mode])
        per_seed.to_csv(f"{a.out}/ppo_{mode}_per_seed.csv", index=False)
        agg = {c: per_seed[c].mean() for c in per_seed.columns if c not in ("policy", "dominant_action")}
        agg.update({f"{c}_std": per_seed[c].std() for c in ("success", "reward", "dominant_share", "action_entropy")})
        agg["policy"] = f"ppo_{mode} (mean over {len(runs)} seeds)"
        agg["dominant_action"] = per_seed["dominant_action"].mode()[0]
        rows.append(agg)
        pooled = pd.concat(runs)
        pf.append(per_fault(pooled, f"ppo_{mode}"))
        if mode == "shaped":                      # paired comparison vs heuristic, per episode
            ppo_ep = pd.concat(runs).groupby("seed")["success"].mean().loc[test_seeds].values
            for other in ("heuristic_rules", "random", "lookahead2_model_based"):
                d, lo, hi = paired_bootstrap_diff(ppo_ep, results[other].set_index("seed").loc[test_seeds, "success"].astype(float).values)
                ppo_summ.append({"comparison": f"ppo_shaped - {other}", "mean_success_diff": d,
                                 "ci_lo": lo, "ci_hi": hi})
    summary = pd.DataFrame(rows)
    summary.to_csv(f"{a.out}/summary.csv", index=False)
    pd.concat(pf).to_csv(f"{a.out}/per_fault_type.csv", index=False)
    if ppo_summ:
        pd.DataFrame(ppo_summ).to_csv(f"{a.out}/paired_differences.csv", index=False)
    cols = ["policy", "success", "ci_lo", "ci_hi", "steps_to_recover", "reward",
            "harmful_actions_per_ep", "dominant_action", "dominant_share", "action_entropy"]
    print(summary[cols].round(3).to_string(index=False))
    print("\nPer fault type:\n", pd.concat(pf).round(3).to_string(index=False))
    if ppo_summ:
        print("\nPaired differences:\n", pd.DataFrame(ppo_summ).round(3).to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/nfm_experiments/nfm/detection.py
"""Detection Agent experiments on Captured_packets.csv (real packet records).

Adds, relative to the original paper code:
  * PR curves, PR-AUC, 95% bootstrap CIs (cluster bootstrap by flow)
  * decision threshold chosen on out-of-fold training predictions (not on test data)
  * prevalence-adjusted precision table
  * temporal drift: metrics and prevalence in consecutive chronological windows
  * feature-causality check (flow-constant features = possible look-ahead aggregates)
    and an ablation without them
  * label audit: how much of `spike_anomaly` is a simple function of the features
  * optional independent/external capture (--external_csv)

Run:  python -m nfm.detection --csv Captured_packets.csv --out results/detection
"""
import argparse
import os

import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

from .common import bootstrap_ci, set_seed

FEATURES = ["packet_length", "tcp_src_port", "tcp_dst_port", "ttl", "tcp_flags", "window_size",
            "ack_rtt", "retransmission", "time_delta", "avg_latency", "jitter", "total_data",
            "session_duration", "bandwidth"]
KEY = ["src_ip", "dst_ip", "protocol", "tcp_src_port", "tcp_dst_port"]


def make_model(spw):
    try:
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, eval_metric="logloss", random_state=42,
                             scale_pos_weight=spw, n_jobs=-1)
    except ImportError:                      # only so the script can be smoke-tested without xgboost
        from sklearn.ensemble import HistGradientBoostingClassifier
        print("WARNING: xgboost not installed, using sklearn fallback (NOT the paper's model)")
        return HistGradientBoostingClassifier(max_iter=200, max_depth=6, learning_rate=0.05,
                                              random_state=42, class_weight={0: 1.0, 1: spw})


def load(path):
    df = pd.read_csv(path)
    t = pd.to_datetime(df["time"], errors="coerce")
    if t.isna().all():
        t = pd.to_datetime(pd.to_numeric(df["time"], errors="coerce"), unit="s")
    df["_t"] = t
    df["_flow"] = df[KEY].astype(str).agg("|".join, axis=1)
    X = df[FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return df, X, df["spike_anomaly"].astype(int).values


def get_splits(df, y):
    idx = np.arange(len(df))
    rtr, rte = train_test_split(idx, test_size=0.2, stratify=y, random_state=42)
    order = np.argsort(df["_t"].values, kind="stable")
    cut = int(0.8 * len(order))
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    ftr, fte = next(gss.split(idx, y, groups=df["_flow"].values))
    return {"random": (rtr, rte), "time_aware": (order[:cut], order[cut:]), "flow_aware": (ftr, fte)}


def best_f1_threshold(y, p):
    pr, rc, th = precision_recall_curve(y, p)
    f1 = 2 * pr * rc / np.maximum(pr + rc, 1e-12)
    return float(th[np.nanargmax(f1[:-1])]) if len(th) else 0.5


def fit_predict(Xtr, ytr, gtr, Xte, cols):
    med = Xtr[cols].median()
    A, B = Xtr[cols].fillna(med).values, Xte[cols].fillna(med).values
    spw = (ytr == 0).sum() / max((ytr == 1).sum(), 1)
    p = make_model(spw).fit(A, ytr).predict_proba(B)[:, 1]
    oof = np.zeros(len(ytr))
    for tr, va in GroupKFold(3).split(A, ytr, gtr):          # threshold from training data only
        oof[va] = make_model(spw).fit(A[tr], ytr[tr]).predict_proba(A[va])[:, 1]
    return p, best_f1_threshold(ytr, oof)


def metrics(y, p, thr, groups, n_boot):
    pred = (p >= 0.5).astype(int)
    f1_ci = bootstrap_ci(lambda a, b: f1_score(a, (b >= 0.5).astype(int)), y, p, n_boot=n_boot, groups=groups)
    ap_ci = bootstrap_ci(average_precision_score, y, p, n_boot=n_boot, groups=groups)
    tp = int(((pred == 1) & (y == 1)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    return {"n_test": len(y), "prevalence": float(y.mean()), "accuracy": accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred),
            "f1": f1_score(y, pred), "f1_ci_lo": f1_ci[0], "f1_ci_hi": f1_ci[1],
            "roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p),
            "pr_auc_ci_lo": ap_ci[0], "pr_auc_ci_hi": ap_ci[1],
            "threshold_from_train_oof": thr, "f1_at_oof_threshold": f1_score(y, (p >= thr).astype(int)),
            "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "tpr": tp / max(tp + fn, 1), "fpr": fp / max(fp + tn, 1)}


def run_protocols(df, X, y, cols, n_boot, tag="all_features"):
    rows, probs = [], {}
    for name, (tr, te) in get_splits(df, y).items():
        p, thr = fit_predict(X.iloc[tr], y[tr], df["_flow"].values[tr], X.iloc[te], cols)
        m = metrics(y[te], p, thr, df["_flow"].values[te], n_boot)
        rows.append({"features": tag, "split": name, **m})
        probs[name] = (y[te], p, te)
    return pd.DataFrame(rows), probs


def prevalence_table(res):
    out = []
    for _, r in res.iterrows():
        for pi in (0.01, 0.02, 0.05, 0.10):
            prec = r.tpr * pi / max(r.tpr * pi + r.fpr * (1 - pi), 1e-12)
            out.append({"split": r.split, "prevalence": pi, "expected_precision": prec})
    return pd.DataFrame(out)


def drift_table(df, probs, n_windows=5):
    y, p, te = probs["time_aware"]
    order = np.argsort(df["_t"].values[te], kind="stable")
    y, p = y[order], p[order]
    rows = []
    for i, w in enumerate(np.array_split(np.arange(len(y)), n_windows)):
        yy, pp = y[w], p[w]
        pred = (pp >= 0.5).astype(int)
        both = 0 < yy.sum() < len(yy)
        rows.append({"window": i + 1, "n": len(w), "prevalence": yy.mean(),
                     "precision": precision_score(yy, pred, zero_division=0),
                     "recall": recall_score(yy, pred, zero_division=0),
                     "f1": f1_score(yy, pred, zero_division=0),
                     "pr_auc": average_precision_score(yy, pp) if both else np.nan})
    return pd.DataFrame(rows)


def feature_causality(df, X):
    g = X.groupby(df["_flow"])
    multi = g.size()[g.size() >= 2].index
    var_ratio = (g.var().loc[multi].mean() / X.var()).rename("within_flow_var_ratio")
    single = (g.nunique(dropna=True).loc[multi] <= 1).mean().rename("flows_with_single_value")
    t = pd.concat([var_ratio, single], axis=1)
    t["flow_constant_suspect"] = (t.flows_with_single_value > 0.9) & ~t.index.isin(
        ["tcp_src_port", "tcp_dst_port"])
    return t.reset_index().rename(columns={"index": "feature"})


def label_audit(df, X, y):
    tr, te = get_splits(df, y)["time_aware"]
    med = X.iloc[tr].median()
    rows, rules = [], ""
    for d in (1, 2, 3, 5):
        t = DecisionTreeClassifier(max_depth=d, class_weight="balanced", random_state=0)
        t.fit(X.iloc[tr].fillna(med), y[tr])
        pred = t.predict(X.iloc[te].fillna(med))
        rows.append({"tree_depth": d, "f1": f1_score(y[te], pred), "precision": precision_score(y[te], pred, zero_division=0),
                     "recall": recall_score(y[te], pred)})
        if d == 3:
            rules = export_text(t, feature_names=FEATURES)
    return pd.DataFrame(rows), rules


def plot_pr(probs, path):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    plt.figure(figsize=(5, 4))
    for name, (y, p, _) in probs.items():
        pr, rc, _ = precision_recall_curve(y, p)
        plt.plot(rc, pr, label=f"{name} (AP={average_precision_score(y, p):.3f})")
    plt.xlabel("Recall"); plt.ylabel("Precision"); plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
    plt.savefig(path, dpi=200); plt.close()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", required=True)
    ap.add_argument("--external_csv", default=None, help="independent capture, same schema")
    ap.add_argument("--out", default="results/detection")
    ap.add_argument("--n_boot", type=int, default=500)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    set_seed(42)
    df, X, y = load(a.csv)
    print(f"{len(df)} records, prevalence {y.mean():.4f}, {df['_flow'].nunique()} five-tuple flows")

    res, probs = run_protocols(df, X, y, FEATURES, a.n_boot)
    res.to_csv(f"{a.out}/protocols.csv", index=False)
    prevalence_table(res).to_csv(f"{a.out}/prevalence_adjusted_precision.csv", index=False)
    drift = drift_table(df, probs); drift.to_csv(f"{a.out}/temporal_drift.csv", index=False)
    plot_pr(probs, f"{a.out}/pr_curves.png")

    caus = feature_causality(df, X); caus.to_csv(f"{a.out}/feature_causality.csv", index=False)
    suspects = caus.loc[caus.flow_constant_suspect, "feature"].tolist()
    keep = [f for f in FEATURES if f not in suspects]
    abl = pd.DataFrame()
    if suspects:
        abl, _ = run_protocols(df, X, y, keep, a.n_boot, tag="without_flow_constant_features")
        abl.to_csv(f"{a.out}/ablation_no_flow_constant_features.csv", index=False)
    audit, rules = label_audit(df, X, y)
    audit.to_csv(f"{a.out}/label_audit.csv", index=False)
    open(f"{a.out}/label_audit_tree_depth3.txt", "w").write(rules)

    if a.external_csv:
        d2, X2, y2 = load(a.external_csv)
        p, thr = fit_predict(X, y, df["_flow"].values, X2, FEATURES)
        pd.DataFrame([metrics(y2, p, thr, d2["_flow"].values, a.n_boot)]).to_csv(
            f"{a.out}/external_capture.csv", index=False)

    cols = ["split", "prevalence", "precision", "recall", "f1", "f1_ci_lo", "f1_ci_hi", "roc_auc",
            "pr_auc", "pr_auc_ci_lo", "pr_auc_ci_hi", "f1_at_oof_threshold"]
    print(res[cols].round(4).to_string(index=False))
    print("\nTemporal drift (time-aware test set, consecutive windows):\n", drift.round(4).to_string(index=False))
    print("\nFlow-constant (possible look-ahead aggregate) features:", suspects or "none")
    if len(abl):
        print(abl[["features"] + cols].round(4).to_string(index=False))
    print("\nLabel audit (shallow trees, time-aware split):\n", audit.round(4).to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/nfm_experiments/nfm/diagnosis.py
"""Diagnosis Agent experiments (root-cause localisation).

Graphs come from the shared simulator (nfm.sim): 5 fault types with documented signatures
and magnitudes, multiple simultaneous faults, observation noise, missing telemetry, three
topology families and graph sizes from 5 to 100 nodes. The GCN is compared with rule-based
and conventional-ML baselines and reported with top-k accuracy and per-fault-type metrics.

Run:  python -m nfm.diagnosis --out results/diagnosis [--fast]
Requires torch (the GCN is a dense implementation of the Kipf & Welling layer, i.e. the
same math as torch_geometric.nn.GCNConv with self-loops and symmetric normalisation).
"""
import argparse
import os

import numpy as np
import pandas as pd

from .common import set_seed
from .sim import FAULT_TYPES, HEALTH_W, Network

TOPOS = ["random", "ba", "ring"]


# --------------------------------------------------------------------------- data
def make_graph(rng, n, k=1, noise=0.0, p_missing=0.0, topo=None, mag_range=(0.35, 0.60)):
    net = Network(n, rng, topo=topo or str(rng.choice(TOPOS)), noise=noise, p_missing=p_missing)
    net.inject_random(k, mag_range)
    return {"X": net.telemetry(), "adj": net.adj.copy(), "roots": list(net.roots),
            "ftypes": list(net.ftypes)}


def make_set(seed, n_graphs, n_range=(5, 31), k_choices=(1, 2), noise_max=0.05, miss_max=0.2):
    """Training distribution: mixed sizes, 1-2 faults, random noise and missing telemetry."""
    rng = np.random.default_rng(seed)
    return [make_graph(rng, int(rng.integers(*n_range)), int(rng.choice(k_choices)),
                       float(rng.uniform(0, noise_max)), float(rng.uniform(0, miss_max)))
            for _ in range(n_graphs)]


def node_features(g):
    """Per-node features for conventional ML: own metrics, neighbour mean, deficit stats."""
    X, A = g["X"], g["adj"]
    deg = np.maximum(A.sum(1, keepdims=True), 1)
    nb = (A @ X) / deg
    deficit = (X @ HEALTH_W)[:, None]
    nbd = (A @ deficit) / deg
    return np.hstack([X, nb, deficit, deficit - nbd])


def labels(g):
    y = np.zeros(len(g["X"]), dtype=int)
    y[g["roots"]] = 1
    return y


# --------------------------------------------------------------------------- GCN
def build_gcn(din=7, hidden=32):
    import torch
    import torch.nn as nn

    class DenseGCN(nn.Module):
        """2 graph-convolution layers + linear output; node-level binary classification."""

        def __init__(self):
            super().__init__()
            self.w1, self.w2 = nn.Linear(din, hidden), nn.Linear(hidden, hidden)
            self.out = nn.Linear(hidden, 2)

        @staticmethod
        def norm(adj):
            a = adj + torch.eye(adj.shape[0])
            d = a.sum(1)
            return a / torch.sqrt(d[:, None] * d[None, :])

        def forward(self, x, adj):
            a = self.norm(adj)
            h = torch.relu(a @ self.w1(x))
            h = torch.relu(a @ self.w2(h))
            return self.out(h)

    return DenseGCN()


def train_gcn(graphs, seed, epochs=60, lr=0.01, batch=32):
    import torch
    import torch.nn.functional as F
    set_seed(seed)
    model = build_gcn()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    pos = sum(len(g["roots"]) for g in graphs)
    tot = sum(len(g["X"]) for g in graphs)
    w = torch.tensor([1.0, (tot - pos) / max(pos, 1)], dtype=torch.float32)   # class-weighted CE
    T = [(torch.tensor(g["X"], dtype=torch.float32), torch.tensor(g["adj"], dtype=torch.float32),
          torch.tensor(labels(g))) for g in graphs]
    rng = np.random.default_rng(seed)
    losses = []
    for ep in range(epochs):
        order, ep_loss = rng.permutation(len(T)), 0.0
        for i in range(0, len(T), batch):
            opt.zero_grad()
            loss = sum(F.cross_entropy(model(T[j][0], T[j][1]), T[j][2], weight=w)
                       for j in order[i:i + batch]) / len(order[i:i + batch])
            loss.backward()
            opt.step()
            ep_loss += float(loss) * len(order[i:i + batch])
        losses.append(ep_loss / len(T))
    model.eval()
    model.train_losses = losses
    return model


def gcn_scores(model):
    import torch

    def f(g):
        with torch.no_grad():
            z = model(torch.tensor(g["X"], dtype=torch.float32), torch.tensor(g["adj"], dtype=torch.float32))
            return torch.softmax(z, 1)[:, 1].numpy()
    return f


# --------------------------------------------------------------------------- baselines
def rule_deficit(g):                       # node with the largest health deficit
    return g["X"] @ HEALTH_W


def rule_neighbour(g):                     # deficit relative to neighbours (root > propagated)
    d = g["X"] @ HEALTH_W
    A = g["adj"]
    return d - 0.5 * (A @ d) / np.maximum(A.sum(1), 1)


def random_scores(seed):
    rng = np.random.default_rng(seed)
    return lambda g: rng.random(len(g["X"]))


def train_rf(graphs, seed):
    from sklearn.ensemble import RandomForestClassifier
    Xtr = np.vstack([node_features(g) for g in graphs])
    ytr = np.concatenate([labels(g) for g in graphs])
    rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=seed, n_jobs=-1)
    rf.fit(Xtr, ytr)
    return lambda g: rf.predict_proba(node_features(g))[:, 1]


# --------------------------------------------------------------------------- evaluation
def evaluate(graphs, score_fn):
    """Top-k: at least one true root in the k top-ranked nodes. recall@k: fraction of roots in
    top-k. Node-level metrics predict exactly |roots| nodes as faulty (precision == recall)."""
    top = {1: [], 2: [], 3: []}
    rec = {1: [], 3: []}
    tp = n_pred = n_true = 0
    per_type = {}
    for g in graphs:
        s = score_fn(g)
        rank = np.argsort(-s)
        roots = set(g["roots"])
        for k in top:
            top[k].append(len(roots & set(rank[:k])) > 0)
        for k in rec:
            rec[k].append(len(roots & set(rank[:k])) / len(roots))
        kk = len(roots)
        hit = len(roots & set(rank[:kk]))
        tp, n_pred, n_true = tp + hit, n_pred + kk, n_true + kk
        for r, ft in zip(g["roots"], g["ftypes"]):
            d = per_type.setdefault(ft, [0, 0])
            d[0] += int(r in set(rank[:kk])); d[1] += 1
    out = {"top1": np.mean(top[1]), "top2": np.mean(top[2]), "top3": np.mean(top[3]),
           "recall_at_1": np.mean(rec[1]), "recall_at_3": np.mean(rec[3]),
           "node_precision": tp / n_pred, "node_recall": tp / n_true}
    out.update({f"recall_{ft}": per_type[ft][0] / per_type[ft][1] for ft in per_type})
    return out


SCENARIOS = {
    "clean_5-30": dict(n_range=(5, 31), k=1, noise=0.0, miss=0.0),
    "noisy": dict(n_range=(5, 31), k=1, noise=0.05, miss=0.0),
    "missing_30pct": dict(n_range=(5, 31), k=1, noise=0.0, miss=0.3),
    "two_faults": dict(n_range=(8, 31), k=2, noise=0.0, miss=0.0),
    "noisy+missing+2faults": dict(n_range=(8, 31), k=2, noise=0.05, miss=0.2),
    "large_50": dict(n_range=(50, 51), k=1, noise=0.0, miss=0.0),
    "large_100": dict(n_range=(100, 101), k=1, noise=0.0, miss=0.0),
    "weak_faults": dict(n_range=(5, 31), k=1, noise=0.03, miss=0.0, mag=(0.15, 0.30)),
}


def scenario_graphs(name, seed, n):
    sc = SCENARIOS[name]
    rng = np.random.default_rng(seed)
    return [make_graph(rng, int(rng.integers(*sc["n_range"])), sc["k"], sc["noise"], sc["miss"],
                       mag_range=sc.get("mag", (0.35, 0.60))) for _ in range(n)]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="results/diagnosis")
    ap.add_argument("--seeds", type=int, default=5)
    ap.add_argument("--n_train", type=int, default=1500)
    ap.add_argument("--n_test", type=int, default=200)
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--fast", action="store_true")
    a = ap.parse_args()
    if a.fast:
        a.seeds, a.n_train, a.n_test, a.epochs = 2, 200, 50, 20
    os.makedirs(a.out, exist_ok=True)
    rows = []
    for seed in range(a.seeds):
        train = make_set(seed, a.n_train)                       # train seed: seed
        gcn = gcn_scores(train_gcn(train, seed, a.epochs))
        rf = train_rf(train, seed)
        methods = {"gcn": gcn, "random_forest": rf, "rule_max_deficit": rule_deficit,
                   "rule_neighbour_adjusted": rule_neighbour, "random": random_scores(seed)}
        for sc in SCENARIOS:
            test = scenario_graphs(sc, 50_000 + seed, a.n_test)  # test seed disjoint from train
            for m, fn in methods.items():
                rows.append({"seed": seed, "scenario": sc, "method": m, **evaluate(test, fn)})
        print("seed", seed, "done")
    df = pd.DataFrame(rows)
    df.to_csv(f"{a.out}/all_runs.csv", index=False)
    num = [c for c in df.columns if c not in ("seed", "scenario", "method")]
    agg = df.groupby(["scenario", "method"])[num].agg(["mean", "std"])
    agg.to_csv(f"{a.out}/summary_mean_std.csv")
    print(df.groupby(["scenario", "method"])[["top1", "top3", "node_precision"]].mean().round(3).to_string())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/nfm_experiments/nfm/integrated.py
"""Integrated closed-loop evaluation (all four agents on ONE common network state).

Interfaces (typed, see dataclasses below):
    Telemetry  : X (n,7) observed metrics in [0,1] (noisy / imputed), adj (n,n) topology
    Detection  : per-node anomaly scores, fired flag
    Diagnosis  : per-node root-cause scores, node = argmax (top-1 target)
    Proposal   : target node + ranked list of recovery actions (best first)
    Verdict    : shield decision for one candidate action (risk, ok)
Mapping between stages (answers "how does a detection become a faulty node / an action"):
    detection  -> flags the *network window* (any node score > threshold calibrated on
                  fault-free windows); no node identity is assumed;
    diagnosis  -> GCN over the same telemetry+topology returns a root-cause ranking;
    recovery   -> observation = telemetry of the diagnosed node + neighbour mean + topology
                  flags; actions that are not available at that node (topology/state) are
                  identified by the orchestrator and rejected by the shield.

Run:  python -m nfm.integrated --out results/integrated --episodes 300 [--ppo_path ppo.zip]
"""
import argparse
import os
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd

from .common import set_seed, wilson
from .detection import make_model
from .diagnosis import (make_set, node_features, rule_deficit, train_rf)
from .recovery import HeuristicPolicy
from .shield import ShieldV1, ShieldV2
from .sim import HEALTH_W, RULE_HEALTH_THR, Network

ADVERSARIAL = ["invalid_id", "unavailable", "disruptive"]


@dataclass
class Telemetry:
    X: np.ndarray
    adj: np.ndarray
    t: int


@dataclass
class Proposal:
    node: int
    ranked: list


@dataclass
class Verdict:
    ok: bool
    risk: float
    attempts: int
    rejections: int
    escalated: bool
    action: int


# --------------------------------------------------------------------------- agents
class LearnedDetector:
    """XGBoost on node features of the simulator telemetry (trained on simulator windows;
    the packet-trace detector of detection.py is evaluated separately on real data)."""

    def __init__(self, seed, n_train=800, n_val=200):
        rng = np.random.default_rng(seed)

        def gen(n_graphs):
            out = []
            for _ in range(n_graphs):
                net = Network(int(rng.integers(5, 31)), rng, topo=str(rng.choice(["random", "ba", "ring"])),
                              noise=0.03, p_missing=0.1)
                if rng.random() < 0.6:
                    net.inject_random(int(rng.choice([1, 2])))
                out.append((net.telemetry(), net.adj.copy(), net.abnormal_nodes().astype(int), net))
            return out

        tr = gen(n_train)
        Xf = np.vstack([node_features({"X": t, "adj": a}) for t, a, _, _ in tr])
        y = np.concatenate([l for _, _, l, _ in tr])
        self.model = make_model((y == 0).sum() / max((y == 1).sum(), 1)).fit(Xf, y)
        val = [net for *_, net in (g for g in gen(n_val)) if not net.roots]
        mx = [self.scores(net.telemetry(), net.adj).max() for net in val]
        self.thr = float(np.quantile(mx, 0.98)) if mx else 0.5     # ~2% window-level false alarms

    def scores(self, X, adj):
        return self.model.predict_proba(node_features({"X": X, "adj": adj}))[:, 1]

    def __call__(self, tel, t=0):
        s = self.scores(tel.X, tel.adj)
        return s, bool(s.max() > self.thr)


class RuleDetector:
    def __call__(self, tel, t=0):
        s = tel.X @ HEALTH_W
        return s, bool((1 - s < RULE_HEALTH_THR).any())


class ScoreDiagnoser:
    def __init__(self, fn):
        self.fn = fn

    def __call__(self, tel):
        return self.fn({"X": tel.X, "adj": tel.adj})


def obs_from_tel(net, tel, v, steps_left, max_steps):
    nb = net.neighbors(v)
    nbm = tel.X[nb].mean(0) if len(nb) else tel.X[v]
    ctx = net.context(v)
    return np.concatenate([tel.X[v], nbm, [min(ctx["degree"], 10) / 10.0, float(ctx["available"][3]),
                                           steps_left / max_steps]]).astype(np.float32)


# --------------------------------------------------------------------------- orchestrator
class Pipeline:
    """Orchestrator loop (pseudocode in README):
         telemetry -> detect -> [if fired] diagnose -> propose (ranked) -> shield gate with
         bounded re-planning -> execute -> re-observe."""

    def __init__(self, name, detector, diagnoser, ranker, shield=None, max_replans=3, max_steps=20):
        self.name, self.detector, self.diagnoser, self.ranker = name, detector, diagnoser, ranker
        self.shield, self.K, self.T = shield, max_replans, max_steps
        self.net_ref = None

    def _adversarial(self, net, v, rng):
        kind = str(rng.choice(ADVERSARIAL))
        av = net.available(v)
        if kind == "invalid_id":
            return 7
        if kind == "unavailable" and (~av).any():
            return int(rng.choice(np.flatnonzero(~av)))
        return int(rng.choice([2, 3]))

    def run(self, net, rng, corrupt_p=0.0):
        self.net_ref = net
        fault = len(net.roots) > 0
        roots = set(net.roots)
        r = dict(fault=fault, recovered=net.recovered(), steps=None, fired_ever=False, first_diag_ok=None,
                 rejections=0, escalations=0, harm=0, executed_nonnoop=0, wrong_node_actions=0,
                 corrupted=0, t_det=[], t_diag=[], t_rec=[], t_shield=[], t_decision=[])
        for t in range(self.T):
            tel = Telemetry(net.telemetry(), net.adj, t)
            t0 = time.perf_counter()
            _, fired = self.detector(tel, t)
            t1 = time.perf_counter()
            r["t_det"].append((t1 - t0) * 1e3)
            if not fired:
                if net.recovered():
                    break
                net.apply(0, 0)                      # idle step: time passes, state drifts
                continue
            r["fired_ever"] = True
            dscore = self.diagnoser(tel)
            t2 = time.perf_counter()
            node = int(np.argmax(dscore))
            if r["first_diag_ok"] is None:
                r["first_diag_ok"] = node in roots
            obs = obs_from_tel(net, tel, node, self.T - t, self.T)
            ranked = list(self.ranker(obs, net, node))
            t3 = time.perf_counter()
            if rng.random() < corrupt_p:             # injected faulty proposal from the learner
                ranked = [self._adversarial(net, node, rng)] + [a for a in ranked]
                r["corrupted"] += 1
            chosen, esc, rej = ranked[0], False, 0
            if self.shield is not None:
                ctx, chosen = net.context(node), None
                for a in ranked[:self.K]:
                    ok, _ = self.shield.validate(tel.X[node], a, ctx)
                    if ok:
                        chosen = a
                        break
                    rej += 1
                if chosen is None:                   # repeated rejection -> safe fallback + escalate
                    chosen, esc = 0, True
            t4 = time.perf_counter()
            r["t_diag"].append((t2 - t1) * 1e3); r["t_rec"].append((t3 - t2) * 1e3)
            r["t_shield"].append((t4 - t3) * 1e3); r["t_decision"].append((t4 - t0) * 1e3)
            r["rejections"] += rej
            r["escalations"] += int(esc)
            r["harm"] += int(net.consequence(node, chosen)["harm"])
            if chosen != 0:
                r["executed_nonnoop"] += 1
                if fault and node not in roots:
                    r["wrong_node_actions"] += 1
            net.apply(node, chosen)
            if net.recovered():
                r["steps"] = t + 1
                break
        r["recovered"] = net.recovered()
        if r["recovered"] and r["steps"] is None:
            r["steps"] = 0 if not fault else self.T
        return r


class ManualOperator(Pipeline):
    """Parameterised stand-in for manual remediation: acts only after `delay` steps, finds the
    root cause with probability loc_acc, applies the rule-based policy, no shield.
    delay / loc_acc are ASSUMPTIONS -- replace with operator logs when available."""

    def __init__(self, delay=5, loc_acc=0.9):
        pol = HeuristicPolicy()
        rd = RuleDetector()
        super().__init__(f"manual_operator(delay={delay})",
                         lambda tel, t=0: (lambda s_f: (s_f[0], s_f[1] and t >= delay))(rd(tel, t)),
                         self._diagnose, lambda obs, net, node: pol.rank(obs, None), None)
        self.loc_acc, self.rng = loc_acc, np.random.default_rng(0)

    def _diagnose(self, tel):
        net = self.net_ref
        s = rule_deficit({"X": tel.X, "adj": tel.adj}).copy()
        if net.roots and self.rng.random() < self.loc_acc:
            s[net.roots[0]] += 10
        return s


# --------------------------------------------------------------------------- experiments
def make_episode(seed, k, n=None):
    rng = np.random.default_rng(seed)
    n = n or int(rng.integers(6, 21))
    net = Network(n, rng, topo=str(rng.choice(["random", "ba", "ring"])), noise=0.02, p_missing=0.05)
    if k > 0:
        net.inject_random(k)
    return net, rng


def run_condition(pipe, n_episodes, seed, p_fault=0.8, k=1, corrupt_p=0.0, n=None):
    rows = []
    for i in range(n_episodes):
        has_fault = np.random.default_rng(seed + i + 777).random() < p_fault
        net, _ = make_episode(seed + i, k if has_fault else 0, n)
        rows.append(pipe.run(net, np.random.default_rng(seed + i + 10 ** 6), corrupt_p))
    return pd.DataFrame(rows)


def summarise(df, name):
    f, nf = df[df.fault], df[~df.fault]
    k, nn = int(f.recovered.sum()), len(f)
    lo, hi = wilson(k, nn)
    lat = np.concatenate([x for x in df.t_decision if len(x)]) if df.t_decision.map(len).sum() else np.array([np.nan])
    mean = lambda col: float(np.concatenate([x for x in df[col] if len(x)]).mean()) if df[col].map(len).sum() else np.nan
    return {"condition": name, "n_fault_eps": nn, "n_nofault_eps": len(nf),
            "e2e_success": k / max(nn, 1), "ci_lo": lo, "ci_hi": hi,
            "mttr_steps": f[f.recovered].steps.mean(),
            "false_remediation_rate": nf.executed_nonnoop.gt(0).mean() if len(nf) else np.nan,
            "wrong_node_action_rate": f.wrong_node_actions.gt(0).mean(),
            "fail_missed_detection": (~f.fired_ever).mean(),
            "fail_wrong_localization": (f.first_diag_ok == False).mean(),
            "fail_recovery_after_correct_loc": ((f.first_diag_ok == True) & ~f.recovered).mean(),
            "harmful_actions_per_ep": df.harm.mean(), "rejections_per_ep": df.rejections.mean(),
            "escalations_per_ep": df.escalations.mean(), "corrupted_proposals_per_ep": df.corrupted.mean(),
            "det_ms": mean("t_det"), "diag_ms": mean("t_diag"), "rec_ms": mean("t_rec"),
            "shield_ms": mean("t_shield"), "decision_ms_mean": float(np.nanmean(lat)),
            "decision_ms_p95": float(np.nanpercentile(lat, 95))}


def scaling(pipe, sizes, n_ep, seed):
    rows = []
    for n in sizes:
        t0 = time.perf_counter()
        df = run_condition(pipe, n_ep, seed, p_fault=1.0, n=n)
        s = summarise(df, f"n={n}")
        s.update({"n_nodes": n, "episodes_per_s": n_ep / (time.perf_counter() - t0)})
        rows.append(s)
    return pd.DataFrame(rows)[["n_nodes", "e2e_success", "det_ms", "diag_ms", "rec_ms", "shield_ms",
                               "decision_ms_mean", "decision_ms_p95", "episodes_per_s"]]


def build_agents(seed, ppo_path, fast):
    det = LearnedDetector(seed, n_train=150 if fast else 800, n_val=60 if fast else 200)
    train = make_set(seed, 150 if fast else 1000)
    try:
        from .diagnosis import gcn_scores, train_gcn
        diag_fn, diag_name = gcn_scores(train_gcn(train, seed, 10 if fast else 60)), "gcn"
    except ImportError:
        print("torch not installed -> using RandomForest as diagnoser (NOT the GCN)")
        diag_fn, diag_name = train_rf(train, seed), "random_forest"
    heur = HeuristicPolicy()
    ranker, rec_name = (lambda obs, net, node: heur.rank(obs, None)), "heuristic_rules"
    if ppo_path:
        from stable_baselines3 import PPO
        from .recovery import PPOPolicy
        pp = PPOPolicy(PPO.load(ppo_path, device="cpu"))
        ranker, rec_name = (lambda obs, net, node: pp.rank(obs)), "ppo"
    print(f"agents: detector=xgboost(sim windows) diagnoser={diag_name} recovery={rec_name}")
    return det, ScoreDiagnoser(diag_fn), ranker, rec_name


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="results/integrated")
    ap.add_argument("--episodes", type=int, default=300)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--ppo_path", default=None, help="stable-baselines3 PPO .zip from nfm.recovery")
    ap.add_argument("--fast", action="store_true")
    a = ap.parse_args()
    if a.fast:
        a.episodes = 40
    os.makedirs(a.out, exist_ok=True)
    set_seed(a.seed)
    det, diag, ranker, rec_name = build_agents(a.seed, a.ppo_path, a.fast)
    full = lambda shield, nm: Pipeline(nm, det, diag, ranker, shield)
    rule = Pipeline("rule_pipeline+shieldV2", RuleDetector(),
                    ScoreDiagnoser(rule_deficit), lambda obs, net, node: HeuristicPolicy().rank(obs, None), ShieldV2())
    test_seed = 500_000
    rows = []
    for cp in (0.0, 0.2):                                    # 0.2 = 20% injected faulty proposals
        for nm, pipe in [(f"full[{rec_name}]+shieldV2", full(ShieldV2(), "")),
                         (f"full[{rec_name}]+shieldV1(paper)", full(ShieldV1(), "")),
                         (f"full[{rec_name}] UNSHIELDED", full(None, ""))]:
            rows.append(summarise(run_condition(pipe, a.episodes, test_seed, corrupt_p=cp), f"{nm} | corrupt={cp}"))
    rows.append(summarise(run_condition(rule, a.episodes, test_seed), "rule_pipeline+shieldV2 | corrupt=0.0"))
    rows.append(summarise(run_condition(ManualOperator(5, 0.9), a.episodes, test_seed), "manual_operator(assumed) | corrupt=0.0"))
    rows.append(summarise(run_condition(full(ShieldV2(), ""), a.episodes, test_seed, k=2),
                          f"full[{rec_name}]+shieldV2 | two simultaneous faults"))
    res = pd.DataFrame(rows)
    res.to_csv(f"{a.out}/end_to_end.csv", index=False)
    sc = scaling(full(ShieldV2(), ""), [5, 10, 25, 50] if a.fast else [5, 10, 25, 50, 100, 200],
                 10 if a.fast else 30, test_seed)
    sc.to_csv(f"{a.out}/scaling_latency.csv", index=False)
    pd.set_option("display.width", 250)
    print(res.round(3).T.to_string())
    print("\nLatency / scaling (ms per decision, wall-clock on this machine):\n", sc.round(2).to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
# CELL 10 - CHECK THE PACKAGE LOADS + ENVIRONMENT REPORT
%cd /kaggle/working/nfm_experiments
import json
from nfm.common import env_report
print(json.dumps(env_report('results/environment.json'), indent=1))

## Detection (reviewer comment 6)

In [ ]:
# CELL 11 - DETECTION quick check
!python -m nfm.detection --csv "{CSV}" --n_boot 50 --out results/detection_fast

In [ ]:
# CELL 12 - DETECTION full
!python -m nfm.detection --csv "{CSV}" --n_boot 500 --out results/detection
import pandas as pd
from IPython.display import display, Image
for f in ['protocols','temporal_drift','feature_causality','label_audit','prevalence_adjusted_precision']:
    print('\n==', f); display(pd.read_csv(f'results/detection/{f}.csv').round(4))
display(Image('results/detection/pr_curves.png'))

## Diagnosis (comment 5)

In [ ]:
# CELL 13 - DIAGNOSIS quick check
!python -m nfm.diagnosis --fast --out results/diagnosis_fast

In [ ]:
# CELL 14 - DIAGNOSIS full
!python -m nfm.diagnosis --out results/diagnosis
import pandas as pd
from IPython.display import display
d = pd.read_csv('results/diagnosis/all_runs.csv')
display(d.groupby(['scenario','method'])[['top1','top2','top3','node_precision']].mean().round(3))

## Recovery (comment 3)

In [ ]:
# CELL 15 - RECOVERY quick check
!python -m nfm.recovery --fast --out results/recovery_fast

In [ ]:
# CELL 16 - RECOVERY full (slowest cell: tuning grid + 5 seeds x 200k steps)
!python -m nfm.recovery --out results/recovery
import pandas as pd
from IPython.display import display
for f in ['summary','per_fault_type','paired_differences']:
    print('\n==', f); display(pd.read_csv(f'results/recovery/{f}.csv').round(3))

## Safety shield (comment 4)

In [ ]:
# CELL 17 - SHIELD
!python -m nfm.shield --n_cases 20000 --out results/shield
import pandas as pd
from IPython.display import display
for f in ['summary','per_kind_v1','per_kind_v2','increment_logit']:
    print('\n==', f); display(pd.read_csv(f'results/shield/{f}.csv').round(3))

## Integrated closed loop (comments 2, 4, 9)

In [ ]:
# CELL 18 - INTEGRATED quick check
!python -m nfm.integrated --fast --out results/integrated_fast

In [ ]:
# CELL 19 - INTEGRATED full (uses the PPO model trained in CELL 16)
!python -m nfm.integrated --episodes 300 --ppo_path results/recovery/ppo_shaped_seed0.zip --out results/integrated
import pandas as pd
from IPython.display import display
pd.set_option('display.width', 250)
display(pd.read_csv('results/integrated/end_to_end.csv').round(3).T)
display(pd.read_csv('results/integrated/scaling_latency.csv').round(2))

In [ ]:
# CELL 20 - ZIP RESULTS (download from the Output tab on the right)
!cd /kaggle/working/nfm_experiments && zip -qr /kaggle/working/results.zip results
print('results.zip created in /kaggle/working')